# FreshCheck Swin + DINOv2 Runner

Notebook ???????????????????????????????????????????????? FreshCheck ???? ?:
- ??? CSV split ??? workflow A/B/C/D ???????????
- ?????????? `swin_t` ??? `DINOv2`
- ??? classification ???? ?????? SAM / patch sampling
- ????? confusion matrix ??? metrics ?? Google Drive


In [ ]:
#@title 1) Mount Drive + clone/update repo
from google.colab import drive
from pathlib import Path
import subprocess
import os

drive.mount('/content/drive')

REPO_URL = 'https://github.com/techasit239/DADS7202_PigPicture.git'
BRANCH = 'main'
REPO_DIR = Path('/content/DADS7202_PigPicture')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

%cd /content/DADS7202_PigPicture
!python -m pip install -q -r requirements.txt timm


In [ ]:
#@title 2) Path configuration
from pathlib import Path

ARTIFACTS_ROOT = Path('/content/drive/MyDrive/FreshCheck/artifacts')
EXPERIMENTS_ROOT = ARTIFACTS_ROOT / 'experiments_manual'
MANIFESTS_ROOT = ARTIFACTS_ROOT / 'manifests'
TARGET_SPLIT_ROOT = ARTIFACTS_ROOT / 'target_split'
SWIN_DINO_ROOT = ARTIFACTS_ROOT / 'swin_dinov2'
SWIN_DINO_ROOT.mkdir(parents=True, exist_ok=True)

print('Artifacts root:', ARTIFACTS_ROOT)
print('Swin+DINO root:', SWIN_DINO_ROOT)


In [ ]:
#@title 3) Imports and dataset helpers
import copy
import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from torchvision import transforms as T
from tqdm.auto import tqdm

CLASS_NAMES = ['FRESH', 'HALF_FRESH', 'SPOILED']
LABEL_MAP = {name: idx for idx, name in enumerate(CLASS_NAMES)}
INDEX_TO_CLASS = {idx: name for name, idx in LABEL_MAP.items()}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class CsvClassificationDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        if 'class' not in self.df.columns or 'path' not in self.df.columns:
            raise ValueError(f'{csv_path} must contain path and class columns')
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        image = self.transform(image)
        label = LABEL_MAP[row['class']]
        return image, label


def build_train_transform(img_size=224):
    return T.Compose([
        T.Resize((256, 256)),
        T.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
        T.RandomHorizontalFlip(p=0.5),
        T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def build_eval_transform(img_size=224):
    return T.Compose([
        T.Resize((256, 256)),
        T.CenterCrop(img_size),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def make_loader(csv_path, batch_size, num_workers, train, img_size=224):
    tfm = build_train_transform(img_size) if train else build_eval_transform(img_size)
    ds = CsvClassificationDataset(csv_path, tfm)
    return DataLoader(ds, batch_size=batch_size, shuffle=train, num_workers=num_workers, pin_memory=True)


In [ ]:
#@title 4) Model builders: Swin-T and DINOv2
class DINOv2LinearProbe(nn.Module):
    def __init__(self, backbone_name='dinov2_vits14', num_classes=3, dropout=0.3):
        super().__init__()
        self.backbone = torch.hub.load('facebookresearch/dinov2', backbone_name)
        for p in self.backbone.parameters():
            p.requires_grad = False

        hidden_size = getattr(self.backbone, 'embed_dim', None)
        if hidden_size is None:
            hidden_size = 384
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes),
        )

    def forward(self, x):
        features = self.backbone(x)
        if isinstance(features, dict):
            if 'x_norm_clstoken' in features:
                features = features['x_norm_clstoken']
            elif 'cls_token' in features:
                features = features['cls_token']
            else:
                features = next(iter(features.values()))
        if features.ndim == 3:
            features = features[:, 0]
        return self.head(features)


def build_swin_t(num_classes=3, dropout=0.3):
    weights = models.Swin_T_Weights.IMAGENET1K_V1
    model = models.swin_t(weights=weights)
    in_features = model.head.in_features
    model.head = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes),
    )
    return model


def build_model(model_name, num_classes=3, dropout=0.3):
    if model_name == 'swin_t':
        return build_swin_t(num_classes=num_classes, dropout=dropout)
    if model_name == 'dinov2_vits14':
        return DINOv2LinearProbe(backbone_name='dinov2_vits14', num_classes=num_classes, dropout=dropout)
    raise ValueError(f'Unsupported model: {model_name}')


In [ ]:
#@title 5) Training and evaluation helpers

def make_class_weights(df):
    counts = df['class'].value_counts()
    total = counts.sum()
    return torch.tensor(
        [total / (len(CLASS_NAMES) * counts[name]) for name in CLASS_NAMES],
        dtype=torch.float32,
    )


def evaluate_model(model, loader, criterion, device):
    model.eval()
    loss_sum = 0.0
    total = 0
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='eval', leave=False):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = model(images)
            loss = criterion(logits, labels)
            preds = logits.argmax(1)
            bs = images.size(0)
            loss_sum += loss.item() * bs
            total += bs
            all_labels.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())

    metrics = {
        'loss': loss_sum / max(total, 1),
        'accuracy': accuracy_score(all_labels, all_preds),
        'macro_precision': precision_score(all_labels, all_preds, average='macro', zero_division=0),
        'macro_recall': recall_score(all_labels, all_preds, average='macro', zero_division=0),
        'macro_f1': f1_score(all_labels, all_preds, average='macro', zero_division=0),
        'labels': all_labels,
        'preds': all_preds,
        'report': classification_report(all_labels, all_preds, target_names=CLASS_NAMES, output_dict=True, zero_division=0),
        'confusion_matrix': confusion_matrix(all_labels, all_preds, labels=[0,1,2]).tolist(),
    }
    return metrics


def save_eval_outputs(output_dir, model_name, metrics):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    payload = {
        'model': model_name,
        'results': {
            'accuracy': float(metrics['accuracy']),
            'macro_precision': float(metrics['macro_precision']),
            'macro_recall': float(metrics['macro_recall']),
            'macro_f1': float(metrics['macro_f1']),
            'loss': float(metrics['loss']),
        },
        'confusion_matrix': {
            'labels': CLASS_NAMES,
            'matrix': metrics['confusion_matrix'],
        },
        'classification_report': metrics['report'],
    }
    (output_dir / f'{model_name}_eval.json').write_text(json.dumps(payload, indent=2), encoding='utf-8')

    matrix = np.asarray(metrics['confusion_matrix'], dtype=int)
    pd.DataFrame(matrix, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(output_dir / f'{model_name}_confusion_matrix.csv')

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(matrix, cmap='Blues')
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix: {model_name}')
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, str(matrix[i, j]), ha='center', va='center', color='black')
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(output_dir / f'{model_name}_confusion_matrix.png', dpi=180)
    plt.close(fig)


def train_one_model(model_name, train_csv, val_csv, output_dir, epochs=15, batch_size=16, num_workers=2, lr=None, weight_decay=1e-2, img_size=224, dropout=0.3, seed=42):
    set_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    train_loader = make_loader(train_csv, batch_size=batch_size, num_workers=num_workers, train=True, img_size=img_size)
    val_loader = make_loader(val_csv, batch_size=batch_size, num_workers=num_workers, train=False, img_size=img_size)

    train_df = pd.read_csv(train_csv)
    class_weights = make_class_weights(train_df).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    model = build_model(model_name, num_classes=3, dropout=dropout).to(device)
    if lr is None:
        lr = 1e-4 if model_name == 'swin_t' else 1e-3

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    output_dir = Path(output_dir)
    ckpt_dir = output_dir / 'checkpoints'
    metrics_dir = output_dir / 'metrics'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    metrics_dir.mkdir(parents=True, exist_ok=True)

    best_state = copy.deepcopy(model.state_dict())
    best_f1 = -1.0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        total = 0
        for images, labels in tqdm(train_loader, desc=f'{model_name} train {epoch}/{epochs}', leave=False):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            bs = images.size(0)
            preds = logits.argmax(1)
            running_loss += loss.item() * bs
            running_correct += (preds == labels).sum().item()
            total += bs

        train_loss = running_loss / max(total, 1)
        train_acc = running_correct / max(total, 1)
        val_metrics = evaluate_model(model, val_loader, criterion, device)
        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_metrics['loss'],
            'val_acc': val_metrics['accuracy'],
            'val_macro_f1': val_metrics['macro_f1'],
        })

        if val_metrics['macro_f1'] > best_f1:
            best_f1 = val_metrics['macro_f1']
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, ckpt_dir / f'{model_name}_best.pt')

        scheduler.step()
        print(f"epoch {epoch}: train_acc={train_acc:.4f} val_acc={val_metrics['accuracy']:.4f} val_macro_f1={val_metrics['macro_f1']:.4f}")

    pd.DataFrame(history).to_csv(metrics_dir / f'{model_name}_history.csv', index=False)
    model.load_state_dict(best_state)
    return model


In [ ]:
#@title 6) Choose experiment and models
EXPERIMENT = 'A'  #@param ['A', 'B', 'C']
MODELS = ['swin_t', 'dinov2_vits14']
EPOCHS = 15  #@param {type:'integer'}
BATCH_SIZE = 16  #@param {type:'integer'}
NUM_WORKERS = 2  #@param {type:'integer'}
SEED = 42  #@param {type:'integer'}

if EXPERIMENT == 'A':
    TRAIN_CSV = MANIFESTS_ROOT / 'source_train_manifest.csv'
    VAL_CSV = MANIFESTS_ROOT / 'source_holdout_manifest.csv'
    TARGET_TEST_CSV = TARGET_SPLIT_ROOT / 'target_holdout.csv'
    EXP_DIR = SWIN_DINO_ROOT / 'exp_a'
elif EXPERIMENT == 'B':
    TRAIN_CSV = EXPERIMENTS_ROOT / 'exp_b_train.csv'
    VAL_CSV = EXPERIMENTS_ROOT / 'exp_b_val.csv'
    TARGET_TEST_CSV = TARGET_SPLIT_ROOT / 'target_holdout.csv'
    EXP_DIR = SWIN_DINO_ROOT / 'exp_b'
elif EXPERIMENT == 'C':
    TRAIN_CSV = EXPERIMENTS_ROOT / 'exp_c_train.csv'
    VAL_CSV = EXPERIMENTS_ROOT / 'exp_c_val.csv'
    TARGET_TEST_CSV = TARGET_SPLIT_ROOT / 'target_holdout.csv'
    EXP_DIR = SWIN_DINO_ROOT / 'exp_c'
else:
    raise ValueError('Unsupported experiment')

print('Experiment:', EXPERIMENT)
print('Train CSV:', TRAIN_CSV)
print('Val CSV:', VAL_CSV)
print('Target test CSV:', TARGET_TEST_CSV)
print('Output dir:', EXP_DIR)


In [ ]:
#@title 7) Train selected models
trained_models = {}
for model_name in MODELS:
    print(f'\n=== Training {model_name} ===')
    trained_models[model_name] = train_one_model(
        model_name=model_name,
        train_csv=TRAIN_CSV,
        val_csv=VAL_CSV,
        output_dir=EXP_DIR,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        seed=SEED,
    )


In [ ]:
#@title 8) Evaluate selected models on Thai target holdout
criterion = nn.CrossEntropyLoss()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
target_loader = make_loader(TARGET_TEST_CSV, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, train=False)

for model_name in MODELS:
    print(f'\n=== Evaluating {model_name} on target holdout ===')
    model = build_model(model_name, num_classes=3).to(device)
    ckpt_path = EXP_DIR / 'checkpoints' / f'{model_name}_best.pt'
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    metrics = evaluate_model(model, target_loader, criterion, device)
    save_eval_outputs(EXP_DIR / 'eval_target', model_name, metrics)
    print(f"{model_name}: accuracy={metrics['accuracy']:.4f} precision={metrics['macro_precision']:.4f} recall={metrics['macro_recall']:.4f} macro_f1={metrics['macro_f1']:.4f} loss={metrics['loss']:.4f}")


In [ ]:
#@title 9) Optional: evaluate Experiment C models on source holdout (Method D)
criterion = nn.CrossEntropyLoss()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
source_holdout_csv = MANIFESTS_ROOT / 'source_holdout_manifest.csv'
source_loader = make_loader(source_holdout_csv, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, train=False)
exp_c_dir = SWIN_DINO_ROOT / 'exp_c'

for model_name in MODELS:
    ckpt_path = exp_c_dir / 'checkpoints' / f'{model_name}_best.pt'
    if not ckpt_path.exists():
        print(f'skip {model_name}: missing checkpoint {ckpt_path}')
        continue
    print(f'\n=== Evaluating {model_name} on source holdout ===')
    model = build_model(model_name, num_classes=3).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    metrics = evaluate_model(model, source_loader, criterion, device)
    save_eval_outputs(exp_c_dir / 'eval_source', model_name, metrics)
    print(f"{model_name}: accuracy={metrics['accuracy']:.4f} precision={metrics['macro_precision']:.4f} recall={metrics['macro_recall']:.4f} macro_f1={metrics['macro_f1']:.4f} loss={metrics['loss']:.4f}")
